In [1]:
import threading
import time


# Multithreading Example in Python


In [ ]:
# -------------------------------------------------------------

# 1. Functions:
#   - f1(): loops 0-4, sleeps 3 seconds per iteration, prints i*i
#   - f2(): loops 0-4, sleeps 3 seconds per iteration, prints i*10
#
# 2. Threads:
#   - t1 = threading.Thread(target=f1) → thread to run f1
#   - t2 = threading.Thread(target=f2) → thread to run f2
#
# 3. Start and join threads:
#   - t1.start(), t2.start() → start both threads concurrently
#   - t1.join(), t2.join() → wait for both threads to finish
#
# 4. Timing:
#   - t = time.time() → record start time
#   - print(time.time() - t) → total time taken
#
# 5. Key Points:
#   - Without threads, each function would run sequentially:
#         5 iterations * 3 seconds * 2 functions = ~30 seconds
#   - With threads, f1 and f2 run concurrently:
#         total time ≈ max(5*3, 5*3) = ~15 seconds
#   - Python threads are suitable for **I/O-bound tasks**.
#     For CPU-bound tasks, use multiprocessing due to the GIL.
#   - The order of printed numbers may **interleave**, e.g., you may see:
#         0
#         0
#         1
#         10
#     etc., because threads run concurrently.
# -------------------------------------------------------------

#funtions
def f1():
  
    for i in range(5):
        time.sleep(3)
        print(i*i)

def f2():
    
    for i in range(5):
        time.sleep(3)
        print(i*10)

#threads
t1=threading.Thread(target=f1)
t2=threading.Thread(target=f2)

t=time.time()
t1.start();t2.start()
t1.join();t2.join()
print(time.time()-t)

00

101

20
4
930

16
40
15.020951271057129


# Multithreading with ThreadPoolExecutor

 ThreadPoolExecutor creates a pool of worker threads that can
 run tasks concurrently. This is useful for I/O-bound tasks
 like network requests, file reading, APIs, etc.
## In this example:
 - numbers is a list of tasks
 - print_number() is the function each thread runs
 - max_workers=3 means at most 3 threads run at the same time

## Execution behavior:
 - First 3 numbers start immediately
 - When a thread finishes (after sleep(1)), it takes the next task
 - So tasks run in batches of 3

### Time expectation:
len(numbers) = 13
workers = 3
each task takes ~1 second
total time ≈ ceil(13 / 3) ≈ 5 seconds

executor.map(function, iterable)
 --------------------------------
 Similar to Python's built-in map(), but executed in parallel.

 Return type:
 - executor.map() returns an **iterator** (not a list)
 - Results are produced lazily as tasks complete

### Example:
results = executor.map(print_number, numbers)

 If print_number returns something:
 results would be an iterator of return values

### Example usage:
for r in results:
    print(r)

## Converting to list:
list(results)

## Should we wrap in list?
 - If you need ALL results immediately → use list(executor.map(fn,lst))
 - If results are large → keep it as iterator to save memory

## Best practice:
 - If the function only performs side effects (like printing),
   you usually don't store the results.
 - Just call executor.map(...) like in this code.

## Why ThreadPoolExecutor?
 - Simplifies thread management
 - Automatically handles thread creation and cleanup
 - Better than manually managing threading.Thread objects

## Significance of max_workers:
 - Controls how many threads run simultaneously
 - Too few → slower
 - Too many → overhead and context switching
 - Default is usually min(32, os.cpu_count() + 4)

## Rule of thumb:
 - I/O tasks → more threads (10–100 sometimes)
 - CPU tasks → use multiprocessing instead

In [20]:
from concurrent.futures import ThreadPoolExecutor
x=time.time()
def print_number(number):
    time.sleep(1)
    print (f"Number :{number}")

numbers=[1,2,3,4,5,6,7,8,9,0,1,2,3]

with ThreadPoolExecutor(max_workers=3) as executor:
    executor.map(print_number,numbers)

print(time.time()-x)

Number :1
Number :2
Number :3
Number :6Number :4
Number :5

Number :7Number :8
Number :9

Number :0Number :2

Number :1
Number :3
5.021286964416504


## Real-World Example: Multithreading for I/O-bound Tasks
### Scenario: Web Scraping
Web scraping often involves making numerous network requests to 
fetch web pages. These tasks are I/O-bound because they spend a lot of
time waiting for responses from servers. Multithreading can significantly
improve the performance by allowing multiple web pages to be fetched concurrently.

In [ ]:

from concurrent.futures import ThreadPoolExecutor

import requests
from bs4 import BeautifulSoup

urls=[
'https://python.langchain.com/v0.2/docs/introduction/',

'https://python.langchain.com/v0.2/docs/concepts/',

'https://python.langchain.com/v0.2/docs/tutorials/'

]

def fetch_content(url):
    response=requests.get(url)
    soup=BeautifulSoup(response.content,'html.parser')
    print(f'Fetched {len(soup.text)} characters from {url}')


with ThreadPoolExecutor(max_workers=3) as executor:
    executor.map(fetch_content,urls)


print("All web pages fetched")

Fetched 4307 characters from https://python.langchain.com/v0.2/docs/concepts/
Fetched 4307 characters from https://python.langchain.com/v0.2/docs/introduction/
Fetched 4307 characters from https://python.langchain.com/v0.2/docs/tutorials/
All web pages fetched
